# UCI BNN — results inspection

Loads the per-split `.pt` files written by `sazz.scripts.uci_bnn` and:
1. Prints a metrics table for one (dataset, split)
2. Plots predictive mean + ±3σ band and per-x epistemic std for 1D toys

Just change `DATASET` and `SPLIT_ID` to inspect a different run.

## Setup

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

if Path.cwd().name == "notebooks":
    os.chdir("..")

from sazz.scripts.uci_bnn import build_target
from sazz.models.bnn_torch import predict_regression

torch.set_default_dtype(torch.float64)

# ---- Pick what to inspect ----
DATASET     = "boston"   # boston / energy / naval / hernandez / gap / sharp / multiscale
SPLIT_ID    = 1
RESULTS_DIR = Path("results/uci_bnn")
#TOY_DIR     = Path("datasets/toy_1d")

split_dir = RESULTS_DIR / DATASET / f"split_{SPLIT_ID:02d}"
print(f"Looking in {split_dir}")
print(f"  found: {sorted(p.name for p in split_dir.glob('*.pt'))}")


## Load all sampler runs

Each `.pt` is a self-contained payload: thinned samples, x_ref, layer_sizes,
metrics, etc. We load them into a dict keyed by sampler name.

In [ ]:
def load_runs(split_dir: Path) -> dict[str, dict]:
    runs = {}
    for pt_path in sorted(split_dir.glob("*.pt")):
        name = pt_path.stem
        runs[name] = torch.load(pt_path, weights_only=False)
    return runs

runs = load_runs(split_dir)
print(f"Loaded {len(runs)} samplers: {list(runs)}")

# Quick peek at one payload
example_sampler = list(runs)[0]
print(f"\nKeys in {example_sampler}.pt: {list(runs[example_sampler])}")


## Metrics summary

In [ ]:
import math
from sazz.scripts.uci_bnn import (
    BNNConfig, build_target, load_toy, BASE_SEED,
    load_raw_datasets, make_split, configs_for, TOY_DIR, UCI_DATASETS,
)

@torch.no_grad()
def predictive_summary(samples, target, X_test):
    """Posterior-predictive mean and std at each test point, on the
    standardised scale. Same as the runner's predict_regression but local
    so we can use it for both metrics and plotting."""
    likelihood = target.meta["model"].likelihood
    X_test = X_test.to(dtype=likelihood.X.dtype, device=likelihood.X.device)
    preds = torch.stack([
        likelihood.predict(beta, X_test).squeeze(-1) for beta in samples
    ])  # [n_samples, n_test]
    return preds.mean(0), preds.std(0)


def gaussian_log_lik(y_true, mean, pred_std, noise_std):
    total_std = (pred_std ** 2 + noise_std ** 2).sqrt()
    return (
        -0.5 * ((y_true - mean) / total_std) ** 2
        - total_std.log()
        - 0.5 * math.log(2 * math.pi)
    ).mean()


def ess_per_coord(samples, max_lag=None):
    x = samples - samples.mean(0, keepdim=True)
    n, d = x.shape
    var = (x ** 2).mean(0)
    if max_lag is None:
        max_lag = min(n - 1, 1000)
    rho_sum = torch.zeros(d, dtype=samples.dtype)
    prev_pair = torch.full((d,), float("inf"), dtype=samples.dtype)
    active = torch.ones(d, dtype=torch.bool)
    k = 1
    while k + 1 <= max_lag:
        c_k   = (x[:n - k]     * x[k:]).mean(0)     / var.clamp(min=1e-30)
        c_kp1 = (x[:n - k - 1] * x[k + 1:]).mean(0) / var.clamp(min=1e-30)
        pair = c_k + c_kp1
        kill = active & ((pair <= 0) | (pair >= prev_pair))
        active = active & ~kill
        rho_sum = rho_sum + torch.where(active, pair, torch.zeros_like(pair))
        prev_pair = torch.where(active, pair, prev_pair)
        if not active.any():
            break
        k += 2
    tau = 1.0 + 2.0 * rho_sum
    return n / tau.clamp(min=1.0)


def compute_metrics(samples, target, data, noise_std):
    """Recompute all metrics from the 4k saved samples, not the runner output."""
    mean_pred, std_pred = predictive_summary(samples, target, data["X_test"])
    rmse_std = ((mean_pred - data["y_test"]) ** 2).mean().sqrt()
    log_lik  = gaussian_log_lik(data["y_test"], mean_pred, std_pred, noise_std)
    ess      = ess_per_coord(samples)
    return {
        "rmse_std":      float(rmse_std),
        "rmse_orig":     float(rmse_std) * data["y_std"],
        "log_lik":       float(log_lik),
        "nll_orig":      -float(log_lik) + math.log(data["y_std"]),
        "pred_std_mean": float(std_pred.mean()),
        "ess_min":       float(ess.min()),
        "ess_median":    float(ess.median()),
        "ess_mean":      float(ess.mean()),
    }


def rebuild_target_for_predictions(payload, dataset, split_id):
    """Reconstruct the (data, target, cfg) needed to compute predictives.

    Works for both toy 1D datasets (loaded from datasets/toy_1d/<name>.pt)
    and UCI datasets (re-derived from raw data + the same seed used at
    sampling time). The target rebuild re-runs Adam to find x_ref — slow
    for big nets but cheap for the architectures we use here.

    Returns (target, data, cfg) where data has the same schema regardless
    of source.
    """
    # ----- Toy: load the saved dataset, take its inline config -----
    if (TOY_DIR / f"{dataset}.pt").exists():
        data, cfg = load_toy(dataset)
        target = build_target(data, cfg)
        return target, data, cfg

    # ----- UCI: re-derive the split using the same seed convention -----
    if dataset in UCI_DATASETS:
        raw = load_raw_datasets()
        X, y = raw[dataset]
        seed = BASE_SEED + split_id
        data = make_split(dataset, X, y, seed=seed)
        cfgs = configs_for({dataset: X.shape[1]})
        cfg = cfgs[dataset]

        # If the payload was saved with different config (e.g. you tweaked
        # noise_std after running), prefer the saved values so the target
        # matches what the samples were drawn from.
        cfg.layer_sizes = payload["layer_sizes"]
        cfg.activation  = payload["activation"]
        cfg.noise_std   = payload["noise_std"]

        target = build_target(data, cfg)
        return target, data, cfg

    raise ValueError(f"Unknown dataset: {dataset}")


# Need a target to compute predictive metrics. Rebuild it once from any
# payload (they all share the same dataset and architecture for a given
# (dataset, split)).
first_payload = next(iter(runs.values()))
target, data, cfg = rebuild_target_for_predictions(first_payload, DATASET, SPLIT_ID)

rows = []
for name, payload in runs.items():
    samples = payload["samples"]
    if name == "map":
        # MAP is a single point estimate, not a posterior — skip the chain
        # metrics and only compute predictive ones.
        m = compute_metrics(samples, target, data, cfg.noise_std)
        m["ess_min"] = m["ess_median"] = m["ess_mean"] = float("nan")
    else:
        m = compute_metrics(samples, target, data, cfg.noise_std)
    rows.append({
        "sampler":          name,
        "n_samples":        samples.shape[0],
        "elapsed_sec":      payload.get("elapsed_sec"),
        **m,
    })

df = pd.DataFrame(rows).set_index("sampler")
df.round(4)

In [ ]:
import pandas as pd

metrics_df = pd.read_csv("results/uci_bnn/metrics.csv")

metrics_df["ess_min_sec"] = metrics_df["ess_min"] / metrics_df["elapsed_sec"]
metrics_df["ess_median_sec"] = metrics_df["ess_median"] / metrics_df["elapsed_sec"]

metrics_out = (
    metrics_df.groupby(["dataset", "sampler"])
      .agg(
          rmse_std_mean=("rmse_std", "mean"),
          rmse_std_sd=("rmse_std", "std"),
          rmse_orig_mean=("rmse_orig", "mean"),
          rmse_orig_sd=("rmse_orig", "std"),
          log_lik=("log_lik", "mean"),
          ess_min_sec=("ess_min_sec", "mean"),
          ess_median_sec=("ess_median_sec", "mean"),
      )
      .reset_index()
)

metrics_out